In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import os

In [7]:
# from groq import Groq
# groq_client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )

In [8]:
from openai import OpenAI
groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = os.getenv("GROQ_API_KEY")
)

In [9]:
from google import genai
from google.genai import types
genai_client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [10]:
def llm(prompt):
    response = groq_client.responses.create(
        model="llama-3.3-70b-versatile",
        input= prompt
    )
    return response.output_text

In [11]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm excited you're interested in the course. However, I need a bit more information from you. Could you please provide more details about the course you're referring to, such as its name or topic? That way, I can better assist you with your inquiry.


In [12]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

You can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    url = f"""{url_prefix}{course['path']}"""
    course_response = requests.get(url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
    
print(len(documents))

1349


In [15]:
from minsearch import Index
index = Index(
    text_fields=["section","question","answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [18]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [19]:
def build_context(search_results):
    lines= []
    
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc["answer"])
        lines.append("")
        
    return '\n'.join(lines).strip()

In [20]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context = context
    )
    
    return prompt.strip()

In [21]:
response = groq_client.responses.create(
    input=prompt,
    model="llama-3.3-70b-versatile"
)

In [22]:
input_price = .44 / 1000000
output_price = .67 / 1000000

cost = (
    input_price * response.usage.input_tokens +
    output_price * response.usage.output_tokens 
)

print(f"{cost:.5f}$")

0.00014$


In [24]:
def llm(instructions, user_prompt, model="openai/gpt-oss-20b"):
    message_history = [
        {'role':'system','content': instructions},
        {'role':'user','content':user_prompt}
    ] 
    
    # response = groq_client.responses.create(
    #     model=model,
    #     input=message_history
    # )
    
    response = genai_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [26]:
def rag(query, model="openai/gpt-oss-20b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)
rag = RAGBase(index, groq_client)
rag.rag("I just discovered the course. Can I join now?")

In [ ]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields= ['question','section','answer'],
    keyword_fields=['course'],
    db_path="FAQ.db"
)
index.search("just discovered the course, can i join now?")

In [ ]:
search_tool = {
    'type' : 'function',
    "name" : 'search',
    "description" : "Search the FAQ database for entries matching the given query.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "query" : {
                "type" : "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["query"],
        "additionalproperties" : False
    }
}

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [ ]:
import json

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

response.output_text

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = groq_client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

In [ ]:
def agent_loop(instructions, question, model=model) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = groq_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query: str) -> dict[str,str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {'question':3.0, 'section':.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict,
        filter_dict= filter_dict
    )

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
chat_interface = IPythonChatInterface()
callback= DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface= chat_interface,
    llm_client=OpenAIClient(client=or_client,model="openrouter/free")
)

result = runner.loop(
    prompt="How do I run Olama locally",
    callback=callback,
)

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

In [ ]:
runner.run()